# Providence, Rhode Island: Land Value Tax Policy Analysis

**City of Providence** (Providence County, FIPS 44007). Rhode Island has no county government,
so the City's real-property levy is the whole local property tax on real estate. That includes
the school appropriation, and no fire district or other overlay levy applies. This model covers
**real property only**. Tangible personal property (Class 4, $53.40) is out of scope.

## Scenarios

| | Scenario A (primary) | Scenario B | Reference B0 |
|---|---|---|---|
| Reform | 4:1 split-rate **within each rate class** | **one citywide** 4:1 split-rate that replaces the classified rates | one flat citywide rate, no split |
| Revenue neutrality | per class (7 classes) | citywide | citywide |
| Inter-class rate ratios | preserved | dissolved | dissolved |
| Purpose | the like-for-like LVT shift | the combined deck scenario | separates B into two parts: B0 − current = the **declassification** effect, B − B0 = the **land tilt** |

All existing exemptions are preserved (elderly, veteran, disabled and blind dollar exemptions
apply improvement-first). Owner-occupied (OO) and non-owner-occupied (NOO) stay separate
classes under A. Tax-stabilization-agreement (TSA, RIGL 44-3-9) and 8%-law affordable-housing
parcels pay negotiated or statutory bills rather than rate × value. They are **held at their
current tax outside the solver** in every scenario. Fully exempt parcels are held out of the
model and excluded from the export and charts.

## FY2026 classified rates (Code sec. 21-182, RIGL 44-5-11.18; the FY2027 adopted rates are identical)

| Roll levy code | Class | Rate per $1,000 |
|---|---|---|
| OO01 / NO01 | 1A, one dwelling unit, owner-occupied / non-owner-occupied | 8.40 / 14.60 |
| OO2-5 / NOO2-5 | 1B, 2 to 5 units, OO / NOO | 7.55 / 14.00 |
| C610 | 1C, 6 to 10 units | 26.00 |
| C11 | 1D, 11+ units | 28.50 |
| C01 | 2A, commercial and industrial (Class 3 mixed-use billed at blended rates) | 29.20 |

## Data sources

| What | Source |
|---|---|
| Parcel polygons + CAMA land / building values | City of Providence GIS, "Parcels with CAMA", `Parcel_Zoning_FL` FeatureServer layer 0, `https://services6.arcgis.com/wv9mHoqblhTsnqdG/arcgis/rest/services/Parcel_Zoning_FL/FeatureServer/0` (TaxRollYear 2025 = assessed **12/31/2024** = FY2026 bills) |
| Rate class, exemptions, billed tax | Providence Open Data, "2025 Property Tax Roll" (Socrata `6ub4-iebe`). The portal text says FY2025, but that is stale: the billed amounts reproduce the FY2026 ordinance rates exactly |
| Official revenue | $352,655,566 = FY2026 certified-roll real-property tax billed (gross), deduplicated by P_ID |
| Rates | FY2026 Approved and FY2027 Adopted Ordinance Books; RI DMF "FY 2026 Rhode Island Tax Rates by Class of Property" (assessment date 12/31/2024; statistical update effective 12/31/2024) |
| Demographics | Census ACS 2022 block groups, FIPS 44007 |

## Column mapping

| Concept | Column | Notes |
|---|---|---|
| Land value | `AssessedValueLand` (CAMA) | 100% of market; condo units carry $0, so it is imputed (Section 3) |
| Improvement value | `AssessedValueBuildings` (CAMA) | |
| Total value | `AssessedValueTotal` (CAMA) = `total_assmt` (roll) | agrees on 100% of joined parcels |
| Rate class | `levy_code_1` (roll) | drives the rate and the scenario-A class |
| Use | `short_desc` (roll) | used for the category map; the CAMA `CLASS` is blank on about 4,600 rows |
| Dollar exemption | `total_exempt` (roll) | applied improvement-first |
| Full exemption | `levy_code_1 == 'E01'` | |
| Billed tax | `total_taxes` (roll) | `current_tax` |
| Parcel ID | `PROPID` = `tax_map` (plat-lot-unit) | join key |
| Neighborhood | Socrata computed region `providence_neighborhoods` (25) | condo land imputation |

**Assessment ratio:** 100% of market value. **Millage source:** the adopted ordinance, cross-checked against billed tax.

In [ ]:
import sys
import json
import os
import shutil
from pathlib import Path

import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from shapely.geometry import box
from dotenv import load_dotenv

sys.path.insert(0, '../..')
REPO_ROOT = Path('../..').resolve()
load_dotenv(REPO_ROOT / '.env')

from lvt.cloud_utils import get_feature_data_with_geometry
from lvt.lvt_utils import (
    model_split_rate_tax,
    calculate_current_tax,
    calculate_category_tax_summary,
    print_category_tax_summary,
    save_standard_export,
    build_standard_export_frame,
)
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

# Constants
CITY_NAME = 'providence'
STATE_FIPS = '44'
COUNTY_FIPS = '007'
MODEL_TYPE = 'split_rate:4.0'          # Scenario A: 4:1 within each rate class
MODEL_TYPE_B = 'split_rate:4.0'        # Scenario B: single citywide 4:1
LAND_IMPROVEMENT_RATIO = 4.0

OFFICIAL_REVENUE = 352_655_566          # FY2026 certified roll, real property billed (gross)

# FY2026 = FY2027 statutory rates by roll levy code (Code sec. 21-182)
STATUTORY_RATE = {
    'OO01': 8.40, 'NO01': 14.60,
    'OO2-5': 7.55, 'NOO2-5': 14.00,
    'C610': 26.00, 'C11': 28.50, 'C01': 29.20,
}
CLASS_LABEL = {
    'OO01': '1A owner-occupied', 'NO01': '1A non-owner-occupied',
    'OO2-5': '1B owner-occupied', 'NOO2-5': '1B non-owner-occupied',
    'C610': '1C 6-10 units', 'C11': '1D 11+ units', 'C01': '2A commercial/industrial',
}
HELD_CODES = ['TSA', '8LAW', 'R01']      # negotiated / statutory bills: held at current tax
FOLD_CODES = {'PRA': 'NO01'}             # 3 Providence Redevelopment Agency parcels billed at 14.60

# Parcel-map identity columns
PARCEL_ID_COL = 'PROPID'
OWNER_NAME_COL = 'owner_name'
OWNER_ADDRESS_COL = 'site_address'
PARCEL_URL_TEMPLATE = None   # no stable public per-parcel deep link found

DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
REPORT_DIR = REPO_ROOT / 'analysis' / 'reports' / CITY_NAME
REPORT_DIR.mkdir(parents=True, exist_ok=True)

## Section 2: Fetch / load parcel data

Two sources are joined on the plat-lot-unit ID. The **CAMA parcel layer** supplies the polygons
and the land/building split. The **FY2026 tax roll** supplies the rate class (levy code),
dollar exemptions, and the billed tax. Both are cached in `data/`.

In [ ]:
# Step 2a: CAMA parcel layer (geometry + land/building values)
PARCEL_PATH = DATA_DIR / 'parcels.gpq'
if PARCEL_PATH.exists():
    raw = gpd.read_parquet(PARCEL_PATH)
    print(f"Loaded {len(raw):,} CAMA parcel features from cache")
else:
    raw = get_feature_data_with_geometry(
        dataset_name='Parcel_Zoning_FL',
        base_url='https://services6.arcgis.com/wv9mHoqblhTsnqdG/arcgis/rest/services',
        layer_id=0,
        paginate=True,
    )
    raw.to_parquet(PARCEL_PATH)
    print(f"Downloaded and cached {len(raw):,} CAMA parcel features")

print("TaxRollYear:", raw['TaxRollYear'].value_counts(dropna=False).to_dict())

# Step 2b: FY2026 tax roll (Socrata JSON API, which includes the neighborhood computed region)
ROLL_PATH = DATA_DIR / 'roll_fy2026.parquet'
if ROLL_PATH.exists():
    roll = pd.read_parquet(ROLL_PATH)
    print(f"Loaded {len(roll):,} roll rows from cache")
else:
    r = requests.get('https://data.providenceri.gov/resource/6ub4-iebe.json',
                     params={'$limit': 60000}, timeout=300)
    r.raise_for_status()
    roll = pd.DataFrame(r.json())
    roll = roll.drop(columns=[c for c in roll.columns if c == 'property_location'])
    roll.to_parquet(ROLL_PATH)
    print(f"Downloaded and cached {len(roll):,} roll rows")

In [ ]:
# Step 2c: clean + join
for c in ['AssessedValueTotal', 'AssessedValueLand', 'AssessedValueBuildings',
          'AreaSF', 'GrBldgArea', 'NumUnits']:
    raw[c] = pd.to_numeric(raw[c], errors='coerce')

g = raw[raw['PROPID'].fillna('').str.len() > 0].copy()
# Multi-part parcels appear as several rows with identical attributes: dissolve them by PROPID
dup = g['PROPID'].duplicated(keep=False)
g_multi = g[dup].dissolve(by='PROPID', aggfunc='first').reset_index()
g = pd.concat([g[~dup], g_multi], ignore_index=True)
g = gpd.GeoDataFrame(g, geometry='geometry', crs=raw.crs)
print(f"CAMA: {len(raw):,} features -> {len(g):,} unique PROPIDs ({dup.sum():,} multi-part rows dissolved)")

roll = roll.copy()
for c in ['total_assmt', 'total_exempt', 'total_taxes']:
    roll[c] = pd.to_numeric(roll[c], errors='coerce').fillna(0.0)
# The roll repeats some P_IDs with identical values (for example, co-owner rows): keep one per P_ID
n_roll_rows = len(roll)
roll = roll.drop_duplicates('p_id').copy()
print(f"Roll: {n_roll_rows:,} rows -> {len(roll):,} parcels after P_ID dedupe; "
      f"billed ${roll['total_taxes'].sum():,.0f}")
assert abs(roll['total_taxes'].sum() - OFFICIAL_REVENUE) < 1, "deduped roll no longer reproduces the official figure"

nbhd_col = [c for c in roll.columns if 'ueqj_g33x' in c]
roll['nbhd'] = pd.to_numeric(roll[nbhd_col[0]], errors='coerce') if nbhd_col else np.nan

def _owner(r):
    if isinstance(r.get('company'), str) and r['company'].strip():
        return r['company'].strip()
    parts = [r.get('first_name'), r.get('last_name')]
    return ' '.join(p for p in parts if isinstance(p, str)).strip() or None
roll['owner_name'] = roll.apply(_owner, axis=1)
roll['site_address'] = roll.get('formated_address')

keep_roll = ['p_id', 'tax_map', 'short_desc', 'levy_code_1', 'total_assmt', 'total_exempt',
             'total_taxes', 'nbhd', 'owner_name', 'site_address']
gdf = g.merge(roll[keep_roll], left_on='PROPID', right_on='tax_map', how='inner')
unmatched = roll[~roll['tax_map'].isin(g['PROPID'])]
print(f"Joined: {len(gdf):,} parcels.  Roll parcels with no CAMA polygon: {len(unmatched):,} "
      f"(${unmatched['total_assmt'].sum()/1e6:,.1f}M assessed, ${unmatched['total_taxes'].sum()/1e6:,.2f}M tax)")
print("  unmatched by levy code:", unmatched['levy_code_1'].value_counts().to_dict())

agree = (gdf['AssessedValueTotal'] == gdf['total_assmt']).mean()
print(f"CAMA total == roll total on {agree*100:.2f}% of joined parcels (same assessment date check)")
assert agree > 0.99
assert ((gdf['AssessedValueLand'] + gdf['AssessedValueBuildings'] - gdf['AssessedValueTotal']).abs() <= 1).mean() > 0.999
assert len(gdf) > 40_000, "Too few parcels: join may be wrong"

## Section 3: Classify and validate

### Condo land imputation + collapse (Gate 5 hazard)

All ~4,100 condo units (residential and commercial) carry **$0 land**, and 640 of 650 complexes
store every unit as a stacked copy of one polygon. Left as-is, condos would read as all-building
and fall artificially under any land tilt. The fix follows the St. Paul precedent:

1. **Impute land.** Each unit's land = unit total × the **median land share of comparable
   non-condo parcels in the same neighborhood**. For residential condos the comparables are
   improved multi-family parcels (2-5 family, 6-10, 11+). For commercial condos they are
   improved commercial and mixed-use parcels. If a neighborhood has fewer than 10 comparables,
   the citywide median is used. Improvement = total − imputed land, so the total is unchanged.
2. **Collapse** the units into one record per **(complex, rate class, use)**, summing values,
   exemptions and billed tax. Collapsing *within* rate class keeps the owner-occupied and
   non-owner-occupied units of one building in their separate scenario-A classes.

In [ ]:
gdf['is_condo'] = gdf['short_desc'].isin(['Residential Condo', 'Commercial Condo'])
gdf['land_raw'] = gdf['AssessedValueLand'].clip(lower=0)
gdf['bldg_raw'] = gdf['AssessedValueBuildings'].clip(lower=0)
gdf['land_share_raw'] = gdf['land_raw'] / gdf['AssessedValueTotal'].replace(0, np.nan)
print(f"Condo units: {gdf.is_condo.sum():,}; with $0 land: {(gdf.is_condo & (gdf.land_raw == 0)).sum():,}")

RES_COMPS = ['2 -5 Family', 'apt 6-10', 'apt 11+']
COM_COMPS = ['Commercial I', 'Commercial II', 'combo 5U', 'combo 6-10', 'Combo 11+']
_ok = (gdf.land_raw > 0) & (gdf.bldg_raw > 0) & (gdf.levy_code_1 != 'E01')

def _nbhd_medians(comp_mask):
    comps = gdf[comp_mask & _ok]
    by_n = comps.groupby('nbhd')['land_share_raw'].agg(['median', 'size'])
    by_n = by_n[by_n['size'] >= 10]['median']
    return by_n, comps['land_share_raw'].median()

res_by_n, res_city = _nbhd_medians(gdf.short_desc.isin(RES_COMPS))
com_by_n, com_city = _nbhd_medians(gdf.short_desc.isin(COM_COMPS))
print(f"Citywide comparable land share: residential multi-family {res_city:.3f}, commercial {com_city:.3f}")

is_res_condo = gdf.short_desc == 'Residential Condo'
is_com_condo = gdf.short_desc == 'Commercial Condo'
imp_share = pd.Series(np.nan, index=gdf.index)
imp_share[is_res_condo] = gdf.loc[is_res_condo, 'nbhd'].map(res_by_n).fillna(res_city)
imp_share[is_com_condo] = gdf.loc[is_com_condo, 'nbhd'].map(com_by_n).fillna(com_city)
gdf['condo_land_imputed'] = gdf.is_condo & (gdf.land_raw == 0)
gdf['land'] = np.where(gdf.condo_land_imputed, gdf.AssessedValueTotal * imp_share, gdf.land_raw)
gdf['bldg'] = gdf['AssessedValueTotal'] - gdf['land']
print(f"Imputed condo land: ${gdf.loc[gdf.condo_land_imputed, 'land'].sum()/1e6:,.1f}M across "
      f"{gdf.condo_land_imputed.sum():,} units (median imputed share "
      f"{imp_share[gdf.condo_land_imputed].median():.3f})")

# Collapse condo units -> (complex, rate class, use)
gdf['complex_id'] = gdf['PROPID'].str[:8]
c = gdf[gdf.is_condo].copy()
sum_cols = ['AssessedValueTotal', 'land', 'bldg', 'total_assmt', 'total_exempt', 'total_taxes',
            'land_raw', 'bldg_raw', 'AreaSF', 'GrBldgArea']
first_cols = [col for col in gdf.columns if col not in sum_cols + ['geometry']]
agg = {col: 'sum' for col in sum_cols}
agg.update({col: 'first' for col in first_cols})
agg['geometry'] = 'first'
agg['PROPID'] = 'size'
coll = c.groupby(['complex_id', 'levy_code_1', 'short_desc'], as_index=False).agg(
    {k: v for k, v in agg.items() if k not in ['complex_id', 'levy_code_1', 'short_desc']})
coll = coll.rename(columns={'PROPID': 'n_units'})
coll['PROPID'] = coll['complex_id'] + '-' + coll['levy_code_1']
coll['owner_name'] = 'Condominium (' + coll['n_units'].astype(str) + ' units, ' + coll['levy_code_1'] + ')'
coll['condo_land_imputed'] = True
coll = gpd.GeoDataFrame(coll, geometry='geometry', crs=gdf.crs)
gdf['n_units'] = 1
gdf = pd.concat([gdf[~gdf.is_condo], coll], ignore_index=True)
gdf = gpd.GeoDataFrame(gdf, geometry='geometry', crs=raw.crs)
print(f"Condo collapse: {len(c):,} units -> {len(coll):,} complex-class records; parcels now {len(gdf):,}")

In [ ]:
# Step 3b: rate class, exemption flags, held-fixed parcels
gdf['rate_class'] = gdf['levy_code_1'].replace(FOLD_CODES)
gdf['full_exmp'] = ((gdf['rate_class'] == 'E01') |
                    ((gdf['total_assmt'] - gdf['total_exempt'] <= 0) & (gdf['total_taxes'] <= 0))).astype(int)
gdf['held_fixed'] = gdf['rate_class'].isin(HELD_CODES) & (gdf['full_exmp'] == 0)
print("Fully exempt:", int(gdf.full_exmp.sum()), " held fixed (TSA/8LAW/R01):", int(gdf.held_fixed.sum()))
print("Rate classes (modeled):",
      gdf.loc[(gdf.full_exmp == 0) & ~gdf.held_fixed, 'rate_class'].value_counts().to_dict())
unknown = gdf.loc[(gdf.full_exmp == 0) & ~gdf.held_fixed & ~gdf.rate_class.isin(STATUTORY_RATE), 'rate_class'].unique()
assert len(unknown) == 0, f"unmapped rate classes: {unknown}"

# Step 3c: property categories from the roll's use description (NOT the CAMA CLASS field)
USE_MAP = {
    'Single Family': 'Single Family Residential',
    '2 -5 Family': 'Small Multi-Family (2-5 units)',
    'apt 6-10': 'Multi-Family (6-10 units)',
    'apt 11+': 'Large Multi-Family (11+ units)',
    'Residential Condo': 'Condominium',
    'Commercial Condo': 'Office / Commercial Condo',
    'combo 5U': 'Mixed Use', 'combo 6-10': 'Mixed Use', 'Combo 11+': 'Mixed Use',
    'Commercial I': 'Commercial', 'Commercial II': 'Commercial',
    'Industrial': 'Industrial',
    'Utility': 'Other Commercial',
    'Farm Forest': 'Agricultural',
    'Residential Vacant Land': 'Vacant Land', 'CI Vacant Land': 'Vacant Land',
    'Miscellaneous': 'Other Residential',        # 8%-law affordable housing (held fixed)
    '44-3-9 Stb': 'Other Commercial',             # tax stabilization agreements (held fixed)
}
gdf['PROPERTY_CATEGORY'] = gdf['short_desc'].map(USE_MAP).fillna('Other')
sfr = gdf['PROPERTY_CATEGORY'] == 'Single Family Residential'
gdf.loc[sfr & (gdf.rate_class == 'OO01'), 'PROPERTY_CATEGORY'] = 'Single Family Residential — Owner-Occupied'
gdf.loc[sfr & (gdf.rate_class != 'OO01'), 'PROPERTY_CATEGORY'] = 'Single Family Residential — Non-Owner-Occupied'
gdf['occupancy'] = np.where(gdf.rate_class.isin(['OO01', 'OO2-5']), 'owner-occupied',
                            np.where(gdf.rate_class.isin(['NO01', 'NOO2-5']), 'non-owner-occupied', ''))
gdf['vacant_type'] = np.where(gdf.short_desc == 'Residential Vacant Land', 'residential',
                              np.where(gdf.short_desc == 'CI Vacant Land', 'commercial', ''))

# Override: $0 improvement -> Vacant Land (keep its res/commercial flavor from the rate class)
zero_b = (gdf['bldg'] <= 0) & (gdf.full_exmp == 0) & ~gdf.held_fixed & (gdf.PROPERTY_CATEGORY != 'Vacant Land')
gdf.loc[zero_b, 'vacant_type'] = np.where(gdf.loc[zero_b, 'rate_class'] == 'C01', 'commercial', 'residential')
gdf.loc[zero_b, 'PROPERTY_CATEGORY'] = 'Vacant Land'
print(f"$0-improvement override -> Vacant Land: {int(zero_b.sum())} parcels")

In [ ]:
# Step 3d: surface parking (OSM amenity=parking coverage >= 50% and building coverage < 10%),
# limited to commercial-class land (Pueblo precedent). CAMA has no parking use code.
from lvt.transit_utils import fetch_osm_parking
gdf['parking_coverage'] = 0.0
try:
    _bnd = gpd.GeoDataFrame(geometry=[box(*gdf.to_crs(4326).total_bounds)], crs=4326)
    _park = fetch_osm_parking(_bnd, str(DATA_DIR / 'osm_parking.gpkg'), to_crs='EPSG:3857')
    _park_union = _park.geometry.buffer(0).union_all()
    _geom = gdf.to_crs(3857).geometry.buffer(0)
    gdf['parking_coverage'] = (_geom.intersection(_park_union).area
                               / _geom.area.replace(0, np.nan)).fillna(0.0).values
    print(f"{len(_park):,} OSM surface/multi-storey parking polygons in the bounding box")
except Exception as exc:
    print(f"OSM parking unavailable ({exc}); no parking category")

gdf['bldg_coverage'] = (gdf['GrBldgArea'].fillna(0) / gdf['AreaSF'].replace(0, np.nan)).fillna(0.0)
gdf['is_parking_lot'] = ((gdf.parking_coverage >= 0.50) & (gdf.bldg_coverage < 0.10)
                         & (gdf.rate_class == 'C01') & (gdf.full_exmp == 0) & ~gdf.held_fixed
                         & gdf.PROPERTY_CATEGORY.isin(['Vacant Land', 'Commercial', 'Other Commercial',
                                                       'Industrial', 'Mixed Use']))
gdf.loc[gdf.is_parking_lot, 'PROPERTY_CATEGORY'] = 'Transportation - Parking'
print(f"Surface parking lots identified: {int(gdf.is_parking_lot.sum()):,} parcels, "
      f"{gdf.loc[gdf.is_parking_lot, 'AreaSF'].sum()/43560:,.0f} acres")

print()
print(gdf.loc[gdf.full_exmp == 0, 'PROPERTY_CATEGORY'].value_counts().to_string())

## Section 4: Current tax model

`current_tax` = the **billed FY2026 tax** from the certified roll. That keeps every parcel at its
observed bill, including Class 3 mixed-use parcels billed at blended residential/commercial rates
and the negotiated TSA / 8%-law bills. As a check on the rate table, the tax is also **rebuilt**
from the statutory rate × (assessed − exemption) and compared class by class.

In [ ]:
gdf['statutory_rate'] = gdf['rate_class'].map(STATUTORY_RATE).fillna(0.0)
_rev, _, gdf = calculate_current_tax(
    df=gdf,
    tax_value_col='total_assmt',
    millage_rate_col='statutory_rate',
    exemption_col='total_exempt',
    exemption_flag_col='full_exmp',
)
gdf = gdf.rename(columns={'current_tax': 'rebuilt_tax'})
gdf['current_tax'] = gdf['total_taxes'].where(gdf.full_exmp == 0, 0.0)

current_revenue = gdf['current_tax'].sum()
gap_pct = (current_revenue / OFFICIAL_REVENUE - 1) * 100
print(f"Modeled: ${current_revenue:,.0f}   Official: ${OFFICIAL_REVENUE:,.0f}   Gap: {gap_pct:+.2f}%")
print("  (the gap is the roll parcels with no CAMA polygon, listed in Section 2)")
assert abs(gap_pct) < 5.0, f"Revenue gap {gap_pct:.1f}% exceeds threshold"

# Class-level validation against the full roll
roll_cls = roll.assign(rate_class=roll['levy_code_1'].replace(FOLD_CODES)).groupby('rate_class')['total_taxes'].sum()
mod_cls = gdf.groupby('rate_class').agg(parcels=('current_tax', 'size'), modeled=('current_tax', 'sum'),
                                        rebuilt=('rebuilt_tax', 'sum'))
mod_cls['roll'] = roll_cls
mod_cls['gap_vs_roll_pct'] = (mod_cls['modeled'] / mod_cls['roll'] - 1) * 100
mod_cls['rebuilt_vs_billed_pct'] = (mod_cls['rebuilt'] / mod_cls['modeled'].replace(0, np.nan) - 1) * 100
print(mod_cls.drop(index='E01', errors='ignore').round(2).to_string())

In [ ]:
# Taxable values after dollar exemptions, applied improvement-first
ex = gdf['total_exempt'].clip(lower=0)
gdf['taxable_improvement_value'] = (gdf['bldg'] - ex).clip(lower=0)
gdf['taxable_land_value'] = (gdf['land'] - (ex - gdf['bldg']).clip(lower=0)).clip(lower=0)
gdf.loc[gdf.full_exmp == 1, ['taxable_land_value', 'taxable_improvement_value']] = 0.0
gdf['taxable_total'] = gdf['taxable_land_value'] + gdf['taxable_improvement_value']

# Parcel effective rate on the net base (= statutory rate except Class 3 blends)
gdf['effective_rate'] = (gdf['current_tax'] * 1000 / gdf['taxable_total'].replace(0, np.nan))
m = (gdf.full_exmp == 0) & ~gdf.held_fixed
dev = (gdf.loc[m, 'effective_rate'] / gdf.loc[m, 'statutory_rate'] - 1).abs()
print(f"Modeled parcels at their statutory class rate (within 0.5%): {(dev < 0.005).mean()*100:.1f}%")
print("Parcels off-rate by class:", gdf.loc[m][dev >= 0.005].rate_class.value_counts().to_dict())
bad = m & (gdf.taxable_total <= 0) & (gdf.current_tax > 0)
print(f"Parcels billed tax with $0 net base (held at current): {int(bad.sum())}")
gdf.loc[bad, 'held_fixed'] = True

## Section 5: Split-rate models

**Scenario A (within class).** Each rate class *c* is solved separately so that its total is
unchanged. Each parcel's land and building values are weighted by its own rate relative to the
class's statutory rate (1.0 for almost every parcel; below 1.0 for Class 3 blends). That way a
mixed-use parcel keeps its blended discount. For a statutory-rate parcel the reported class
millages are its land and building rates.

**Scenario B (citywide).** One land rate and one building rate for all modeled parcels, set so
the citywide total is unchanged. **B0** is the flat single rate with no split. Then B0 − current
isolates the effect of dissolving the classification, and B − B0 isolates the land tilt.

In [ ]:
n_exempt = int((gdf['full_exmp'] == 1).sum())
held = gdf[(gdf.full_exmp == 0) & gdf.held_fixed].copy()
df = gdf[(gdf.full_exmp == 0) & ~gdf.held_fixed].copy()
print(f"Held out {n_exempt:,} fully-exempt parcels (excluded from model, export, and charts).")
print(f"Held at current tax: {len(held):,} TSA/8%-law parcels, ${held.current_tax.sum():,.0f}")
print(f"Modeled: {len(df):,} parcels, ${df.current_tax.sum():,.0f}")

# Scenario A: 4:1 within each class
df['rel_rate'] = (df['effective_rate'] / df['statutory_rate']).fillna(1.0)
df['wl'] = df['taxable_land_value'] * df['rel_rate']
df['wi'] = df['taxable_improvement_value'] * df['rel_rate']
class_millage = {}
parts = []
for cls, sub in df.groupby('rate_class'):
    lm, im, rev, res = model_split_rate_tax(
        df=sub, land_value_col='wl', improvement_value_col='wi',
        current_revenue=float(sub['current_tax'].sum()),
        land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
    )
    class_millage[cls] = {'land_millage': lm, 'improvement_millage': im,
                          'current_rate': STATUTORY_RATE[cls], 'revenue': rev,
                          'parcels': len(sub)}
    parts.append(res['new_tax'])
df['new_tax_A'] = pd.concat(parts).reindex(df.index)
assert df['new_tax_A'].notna().all()

cm = pd.DataFrame(class_millage).T
cm['label'] = cm.index.map(CLASS_LABEL)
cm['land_x_current'] = cm['land_millage'] / cm['current_rate']
print(cm[['label', 'parcels', 'current_rate', 'land_millage', 'improvement_millage', 'land_x_current', 'revenue']]
      .round(3).to_string())
print(f"Scenario A revenue: ${df.new_tax_A.sum():,.0f} vs ${df.current_tax.sum():,.0f}")

# Headline land/improvement millage for the standard export = parcel's own class millage
df['land_millage_A'] = df['rate_class'].map(cm['land_millage']) * df['rel_rate']
df['improvement_millage_A'] = df['rate_class'].map(cm['improvement_millage']) * df['rel_rate']

In [ ]:
# Legal cross-check (information only): RIGL 44-5-11.18 / sec. 21-182(a)(2) caps
#   Class 2A rate <= 2x the Class 1A (non-owner-occupied base) rate
#   Class 2A rate <= 3.5x the effective owner-occupied Class 1A rate
legal = []
for comp in ['land_millage', 'improvement_millage']:
    c2a, c1a, oo = cm.loc['C01', comp], cm.loc['NO01', comp], cm.loc['OO01', comp]
    legal.append({'component': comp.replace('_millage', ''),
                  '2A': round(c2a, 3), '1A_NOO': round(c1a, 3), '1A_OO': round(oo, 3),
                  '2A/1A_NOO': round(c2a / c1a, 3), 'cap_2x_breached': c2a / c1a > 2.0 + 1e-9,
                  '2A/1A_OO': round(c2a / oo, 3), 'cap_3.5x_breached': c2a / oo > 3.5 + 1e-9})
legal = pd.DataFrame(legal)
print("Current: 2A/1A_NOO = {:.3f}, 2A/1A_OO = {:.3f}".format(29.20 / 14.60, 29.20 / 8.40))
print(legal.to_string(index=False))

In [ ]:
# Scenario B0: flat single citywide rate (no split). Scenario B: single citywide 4:1 split.
R = float(df['current_tax'].sum())
flat_millage_B0 = R * 1000 / df['taxable_total'].sum()
df['new_tax_B0'] = df['taxable_total'] * flat_millage_B0 / 1000

land_millage_B, improvement_millage_B, new_revenue_B, _resB = model_split_rate_tax(
    df=df, land_value_col='taxable_land_value', improvement_value_col='taxable_improvement_value',
    current_revenue=R, land_improvement_ratio=LAND_IMPROVEMENT_RATIO,
)
df['new_tax_B'] = _resB['new_tax'].reindex(df.index)
print(f"B0 flat millage: {flat_millage_B0:.4f}")
print(f"B  land millage: {land_millage_B:.4f}   improvement millage: {improvement_millage_B:.4f}")
print(f"Revenue: B0 ${df.new_tax_B0.sum():,.0f}  B ${df.new_tax_B.sum():,.0f}  current ${R:,.0f}")

for s in ['A', 'B', 'B0']:
    df[f'tax_change_{s}'] = df[f'new_tax_{s}'] - df['current_tax']
    df[f'tax_change_pct_{s}'] = np.where(df['current_tax'] > 0,
                                         df[f'tax_change_{s}'] / df['current_tax'] * 100, np.nan)
df['tax_change_declass'] = df['new_tax_B0'] - df['current_tax']     # B0 - current
df['tax_change_tilt'] = df['new_tax_B'] - df['new_tax_B0']           # B - B0

# Canonical columns = Scenario A
df['new_tax'] = df['new_tax_A']
df['tax_change'] = df['tax_change_A']
df['tax_change_pct'] = df['tax_change_pct_A']

cat_A = calculate_category_tax_summary(df=df, category_col='PROPERTY_CATEGORY',
                                       current_tax_col='current_tax', new_tax_col='new_tax_A')
print_category_tax_summary(cat_A, title="Providence — Scenario A: 4:1 within each rate class")
# The summary reads tax_change / tax_change_pct directly, so hand it a frame on Scenario B's basis
_fB = df[['PROPERTY_CATEGORY', 'current_tax', 'new_tax_B', 'tax_change_B', 'tax_change_pct_B']].rename(
    columns={'tax_change_B': 'tax_change', 'tax_change_pct_B': 'tax_change_pct'})
cat_B = calculate_category_tax_summary(df=_fB, category_col='PROPERTY_CATEGORY',
                                       current_tax_col='current_tax', new_tax_col='new_tax_B')
print_category_tax_summary(cat_B, title="Providence — Scenario B: single citywide 4:1")

### Gate 5: artifact scan

In [ ]:
d = df.copy()
d['bldg_share'] = d['bldg'] / d['AssessedValueTotal'].replace(0, np.nan)
scan = d.groupby('PROPERTY_CATEGORY').agg(
    n=('tax_change_pct_A', 'size'),
    median_pct_A=('tax_change_pct_A', 'median'),
    median_pct_B=('tax_change_pct_B', 'median'),
    median_bldg_share=('bldg_share', 'median'),
    pct_zero_bldg=('bldg', lambda s: (s <= 1000).mean()),
    pct_land_imputed=('condo_land_imputed', 'mean'),
).sort_values('median_pct_A', ascending=False)
print(scan.round(3).to_string())

# Base reconciliation: modeled + held + unmatched net base vs the full roll's net base
roll_net = (roll['total_assmt'] - roll['total_exempt']).clip(lower=0)[roll['levy_code_1'] != 'E01'].sum()
mod_net = df['taxable_total'].sum() + (held['total_assmt'] - held['total_exempt']).clip(lower=0).sum()
print(f"\nNet taxable base: modeled+held ${mod_net/1e9:,.3f}B vs roll ${roll_net/1e9:,.3f}B "
      f"({(mod_net/roll_net-1)*100:+.2f}%)")

# Condo imputation sensitivity: condo outcome if the imputed land share were +/-25%
for f in [0.75, 1.25]:
    cc = df[df.condo_land_imputed & (df.PROPERTY_CATEGORY == 'Condominium')]
    L = cc.taxable_land_value * f
    B = cc.taxable_total - L
    w = cc.rel_rate
    lm = cc.rate_class.map(cm.land_millage); im = cc.rate_class.map(cm.improvement_millage)
    alt = (L * w * lm + B * w * im) / 1000
    print(f"Condo median change (A) with imputed land x{f}: {((alt / cc.current_tax - 1) * 100).median():+.1f}% "
          f"(class millages held; indicative)")

### Deck numbers (both scenarios)

The redistribution absorbed by the vacant + underbuilt group is measured as that group's
**net** increase as a share of the scenario's **gross** increase (Σ of all positive changes,
which equals Σ of all decreases under revenue neutrality). "Underbuilt" = improvement below 50%
of assessed value. Pays less/more uses a ±$1 threshold.

In [ ]:
DECK_CATS = {
    'OO SFH': df.PROPERTY_CATEGORY == 'Single Family Residential — Owner-Occupied',
    'NOO SFH': df.PROPERTY_CATEGORY == 'Single Family Residential — Non-Owner-Occupied',
    '2-5 family': df.PROPERTY_CATEGORY == 'Small Multi-Family (2-5 units)',
    '2-5 family OO': (df.PROPERTY_CATEGORY == 'Small Multi-Family (2-5 units)') & (df.occupancy == 'owner-occupied'),
    '2-5 family NOO': (df.PROPERTY_CATEGORY == 'Small Multi-Family (2-5 units)') & (df.occupancy == 'non-owner-occupied'),
    '6-10 units': df.PROPERTY_CATEGORY == 'Multi-Family (6-10 units)',
    '11+ units': df.PROPERTY_CATEGORY == 'Large Multi-Family (11+ units)',
    'Condos (residential)': df.PROPERTY_CATEGORY == 'Condominium',
    'Commercial condos': df.PROPERTY_CATEGORY == 'Office / Commercial Condo',
    'Commercial': df.PROPERTY_CATEGORY == 'Commercial',
    'Mixed use': df.PROPERTY_CATEGORY == 'Mixed Use',
    'Industrial': df.PROPERTY_CATEGORY == 'Industrial',
    'Vacant residential': (df.PROPERTY_CATEGORY == 'Vacant Land') & (df.vacant_type == 'residential'),
    'Vacant commercial': (df.PROPERTY_CATEGORY == 'Vacant Land') & (df.vacant_type == 'commercial'),
    'Surface parking': df.PROPERTY_CATEGORY == 'Transportation - Parking',
}
impr_share = (df['bldg'] / df['AssessedValueTotal'].replace(0, np.nan)).fillna(0)
underbuilt = (df['bldg'] <= 0) | (impr_share < 0.50)

def scen_numbers(s):
    new, chg, pct = df[f'new_tax_{s}'], df[f'tax_change_{s}'], df[f'tax_change_pct_{s}']
    out = {'categories': {}}
    for k, msk in DECK_CATS.items():
        if msk.sum() == 0:
            continue
        out['categories'][k] = {
            'n': int(msk.sum()),
            'median_pct': round(float(pct[msk].median()), 1),
            'aggregate_pct': round(float((new[msk].sum() / df.current_tax[msk].sum() - 1) * 100), 1),
            'net_change_usd': round(float(chg[msk].sum())),
        }
    gross_up = chg.clip(lower=0).sum()
    out['pct_pay_less'] = round(float((chg < -1).mean() * 100), 1)
    out['pct_pay_more'] = round(float((chg > 1).mean() * 100), 1)
    out['gross_redistribution_usd'] = round(float(gross_up))
    out['vacant_underbuilt'] = {
        'n': int(underbuilt.sum()),
        'net_change_usd': round(float(chg[underbuilt].sum())),
        'share_of_gross_increase_pct': round(float(chg[underbuilt].sum() / gross_up * 100), 1),
        'median_pct': round(float(pct[underbuilt].median()), 1),
    }
    return out

deck = {'A': scen_numbers('A'), 'B': scen_numbers('B'), 'B0': scen_numbers('B0')}

# Decomposition of B into declassification (B0 - current) and land tilt (B - B0)
decomp = {}
for k, msk in DECK_CATS.items():
    if msk.sum() == 0:
        continue
    base = df.current_tax[msk].sum()
    decomp[k] = {
        'aggregate_pct_B': round(float(df.tax_change_B[msk].sum() / base * 100), 1),
        'from_declassification_pct': round(float(df.tax_change_declass[msk].sum() / base * 100), 1),
        'from_land_tilt_pct': round(float(df.tax_change_tilt[msk].sum() / base * 100), 1),
    }
deck['B_decomposition'] = decomp

# Land-share diagnostic
sfh = df[df.PROPERTY_CATEGORY.str.startswith('Single Family')]
ls = lambda x: float(x.land.sum() / x.AssessedValueTotal.sum())
deck['land_share'] = {
    'citywide_value_weighted': round(ls(df), 3),
    'citywide_raw_before_condo_imputation': round(float(df.land_raw.sum() / df.AssessedValueTotal.sum()), 3),
    'median_sfh': round(float((sfh.land / sfh.AssessedValueTotal).median()), 3),
    'median_sfh_OO': round(float((sfh.land / sfh.AssessedValueTotal)[sfh.rate_class == 'OO01'].median()), 3),
    'median_sfh_NOO': round(float((sfh.land / sfh.AssessedValueTotal)[sfh.rate_class == 'NO01'].median()), 3),
    'by_class_value_weighted': {c: round(ls(df[df.rate_class == c]), 3) for c in STATUTORY_RATE},
}
deck['millage'] = {
    'A_by_class': {c: {'current': STATUTORY_RATE[c], 'land': round(float(cm.loc[c, 'land_millage']), 3),
                       'improvement': round(float(cm.loc[c, 'improvement_millage']), 3)} for c in cm.index},
    'B': {'land': round(float(land_millage_B), 3), 'improvement': round(float(improvement_millage_B), 3)},
    'B0_flat': round(float(flat_millage_B0), 3),
}
deck['legal_cross_check_A'] = legal.to_dict(orient='records')
deck['counts'] = {'modeled': int(len(df)), 'held_fixed': int(len(held)),
                  'held_fixed_tax_usd': round(float(held.current_tax.sum())),
                  'fully_exempt': n_exempt, 'condo_units_collapsed': int(len(c)),
                  'condo_records': int(len(coll)), 'roll_unmatched': int(len(unmatched)),
                  'modeled_current_revenue_usd': round(float(current_revenue)),
                  'revenue_gap_pct': round(float(gap_pct), 2)}
with open(REPORT_DIR / 'deck_numbers.json', 'w') as f:
    json.dump(deck, f, indent=2, default=str)
print(json.dumps(deck, indent=1, default=str)[:6000])

## Section 6: Exploration charts (optional)

A scenario-B decomposition chart: how much of each category's change comes from dissolving the
classified rates (B0 − current) and how much from the land tilt (B − B0).

In [ ]:
_dc = pd.DataFrame(deck['B_decomposition']).T
_dc = _dc.drop(index=[k for k in ['2-5 family OO', '2-5 family NOO'] if k in _dc.index])
_dc = _dc.sort_values('aggregate_pct_B')
fig, ax = plt.subplots(figsize=(10, 7))
y = np.arange(len(_dc))
ax.barh(y, _dc['from_declassification_pct'], color='#9E9E9E', label='Dissolving the classified rates (B0 − current)')
ax.barh(y, _dc['from_land_tilt_pct'], left=_dc['from_declassification_pct'].where(
    np.sign(_dc['from_declassification_pct']) == np.sign(_dc['from_land_tilt_pct']), 0),
    color='#352977', label='4:1 land tilt (B − B0)')
ax.scatter(_dc['aggregate_pct_B'], y, color='black', zorder=3, s=18, label='Scenario B total')
ax.set_yticks(y); ax.set_yticklabels(_dc.index)
ax.axvline(0, color='black', lw=0.8)
ax.set_xlabel('Aggregate % change in category tax')
ax.set_title('Providence Scenario B: where the change comes from')
ax.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.savefig(REPORT_DIR / 'scenario_b_decomposition.png', dpi=150, bbox_inches='tight')
plt.close()
print("wrote scenario_b_decomposition.png")

## Section 7: Census join + export

The primary standard export (`analysis/data/providence.csv`) and the standard report PNGs are
**Scenario A**. The Scenario B and B0 columns are appended after the 16 standard columns (the
Denver precedent). A standard-format Scenario B CSV is written to
`analysis/data/scenarios/providence_scenario_b.csv`; it sits in a subfolder so the cross-city
glob doesn't pick it up. The Scenario B report PNGs carry a `_scenario_b` suffix. Under A,
`land_millage` / `improvement_millage` in the export are **per-parcel** (each parcel's class
millage), not row-constant.

In [ ]:
# Census join — OPTIONAL, must happen before export
import os
import concurrent.futures
from lvt.census_utils import get_census_data_with_boundaries, match_to_census_blockgroups

gdf = df   # the modeled GeoDataFrame carries geometry + identity columns
_fips = STATE_FIPS + COUNTY_FIPS
_census_cols = ['std_geoid', 'median_income', 'minority_pct', 'black_pct']
if not os.getenv('CENSUS_API_KEY'):
    # No key configured — the census join is optional, so skip it cleanly.
    print("Census join: skipped (no CENSUS_API_KEY)")
    for _col in _census_cols:
        gdf[_col] = float('nan')
else:
    try:
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as _ex:
            _future = _ex.submit(get_census_data_with_boundaries, _fips, 2022)
            try:
                census_data, census_gdf = _future.result(timeout=90)
                gdf = match_to_census_blockgroups(gdf, census_gdf)
                # census_gdf already carries demographics — do NOT merge census_data again.
                if 'minority_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'white_pop' in gdf.columns:
                    gdf['minority_pct'] = ((gdf['total_pop'] - gdf['white_pop']) / gdf['total_pop'] * 100).round(2)
                if 'black_pct' not in gdf.columns and 'total_pop' in gdf.columns and 'black_pop' in gdf.columns:
                    gdf['black_pct'] = (gdf['black_pop'] / gdf['total_pop'] * 100).round(2)
                print(f"Census join: {gdf['std_geoid'].notna().mean()*100:.1f}% matched")
            except concurrent.futures.TimeoutError:
                print("Census API timed out — skipping census join")
                for _col in _census_cols:
                    gdf[_col] = float('nan')
    except Exception as e:
        print(f"Census join failed: {e}")
        for _col in _census_cols:
            gdf[_col] = float('nan')

In [ ]:
# Export — gdf must have census columns by this point
# Headline millage passed to the export = the owner-occupied 1A class (the most common parcel);
# overwritten per parcel right after with each parcel's own class millage.
land_millage = float(cm.loc['OO01', 'land_millage'])
improvement_millage = float(cm.loc['OO01', 'improvement_millage'])

from lvt.lvt_utils import save_standard_export
out_df = save_standard_export(
    df=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/data/{CITY_NAME}.csv',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    property_category_col='PROPERTY_CATEGORY',
    current_tax_col='current_tax',
    new_tax_col='new_tax',
    tax_change_col='tax_change',
    tax_change_pct_col='tax_change_pct',
    taxable_land_col='taxable_land_value',
    taxable_improvement_col='taxable_improvement_value',
    exempt_flag_col='full_exmp',
)
out_df['land_millage'] = gdf['land_millage_A'].to_numpy()
out_df['improvement_millage'] = gdf['improvement_millage_A'].to_numpy()

# Primary CSV: 16 standard columns (Scenario A) + appended scenario columns
full = out_df.copy()
full['rate_class'] = gdf['rate_class'].to_numpy()
for s in ['B', 'B0']:
    full[f'new_tax_{s}'] = gdf[f'new_tax_{s}'].to_numpy()
    full[f'tax_change_{s}'] = gdf[f'tax_change_{s}'].to_numpy()
    full[f'tax_change_pct_{s}'] = gdf[f'tax_change_pct_{s}'].to_numpy()
full['land_millage_B'] = land_millage_B
full['improvement_millage_B'] = improvement_millage_B
full['flat_millage_B0'] = flat_millage_B0
full['condo_land_imputed'] = gdf['condo_land_imputed'].astype(bool).to_numpy()
full['n_units'] = gdf['n_units'].to_numpy()
full.to_csv(f'../../analysis/data/{CITY_NAME}.csv', index=False)
print(f"Wrote ../../analysis/data/{CITY_NAME}.csv  {full.shape}")

# Standalone standard-format Scenario B export (subfolder: not picked up by the cross-city glob)
gdf['new_tax_Bstd'] = gdf['new_tax_B']
_scen_dir = REPO_ROOT / 'analysis' / 'data' / 'scenarios'
_scen_dir.mkdir(parents=True, exist_ok=True)
out_b = save_standard_export(
    df=gdf, city=CITY_NAME, output_path=str(_scen_dir / f'{CITY_NAME}_scenario_b.csv'),
    model_type=MODEL_TYPE_B, land_millage=land_millage_B, improvement_millage=improvement_millage_B,
    property_category_col='PROPERTY_CATEGORY', current_tax_col='current_tax',
    new_tax_col='new_tax_B', tax_change_col='tax_change_B', tax_change_pct_col='tax_change_pct_B',
    taxable_land_col='taxable_land_value', taxable_improvement_col='taxable_improvement_value',
    exempt_flag_col='full_exmp',
)

RES_CATS = ['Single Family Residential — Owner-Occupied', 'Single Family Residential — Non-Owner-Occupied',
            'Small Multi-Family (2-5 units)', 'Multi-Family (6-10 units)', 'Large Multi-Family (11+ units)',
            'Condominium', 'Mixed Use']

# Standard report — 7 PNGs in analysis/reports/<city>/ (Scenario A)
from lvt.viz import create_city_report
create_city_report(out_df, CITY_NAME, show=False, census_categories=RES_CATS)

# Scenario B report: same chart set, suffixed _scenario_b in the city report folder
_tmp = DATA_DIR / '_report_b'
if _tmp.exists():
    shutil.rmtree(_tmp)
create_city_report(out_b, f'{CITY_NAME}_scenario_b', output_dir=str(_tmp), show=False, census_categories=RES_CATS)
for p in (_tmp / f'{CITY_NAME}_scenario_b').iterdir():
    stem, ext = p.stem, p.suffix
    if ext == '.png':
        shutil.copy(p, REPORT_DIR / f'{stem}_scenario_b{ext}')
    elif p.name == 'metrics_summary.md':
        shutil.copy(p, REPORT_DIR / 'metrics_summary_scenario_b.md')
shutil.rmtree(_tmp)
print("Scenario B PNGs:", sorted(x.name for x in REPORT_DIR.glob('*_scenario_b.png')))

# Parcel-map export (Scenario A) — GeoParquet + interactive HTML map
from lvt.parcel_map import save_parcel_map_export, create_parcel_map
map_gdf = save_parcel_map_export(
    gdf=gdf,
    city=CITY_NAME,
    output_path=f'../../analysis/maps/{CITY_NAME}.parquet',
    model_type=MODEL_TYPE,
    land_millage=land_millage,
    improvement_millage=improvement_millage,
    parcel_id_col=PARCEL_ID_COL,
    parcel_url_template=PARCEL_URL_TEMPLATE,
    owner_name_col=OWNER_NAME_COL,
    owner_address_col=OWNER_ADDRESS_COL,
)
create_parcel_map(map_gdf, CITY_NAME)
print("Done.")

## Validation summary

| Check | Result |
|---|---|
| Revenue match | −1.05% vs official $352,655,566 (FY2026 certified roll, billed real property). Modeled $348,954,496; the whole gap is the 313 roll parcels with no CAMA polygon. By class: −0.2% (OO01) to −2.9% (C11); every class is within 5% |
| Rate rebuild | Statutory rate × (assessed − exemption) reproduces billed tax to 0.00% in every residential class; C01 is +3.95% because Class 3 mixed-use parcels are billed at blended rates, so billed tax is used |
| Parcel count | 43,749 joined parcels → 37,306 modeled records (4,325 condo units collapsed into 1,083 complex × class records); 2,515 fully exempt parcels held out; 686 TSA / 8%-law parcels held at current tax ($26.99M) |
| Census coverage | 100.0% matched to block groups (83% of rows have non-suppressed income) |
| PNGs generated | 7 of 7 (Scenario A) + 7 `_scenario_b` + `scenario_b_decomposition.png` |
| Parcel map | `analysis/maps/providence.parquet` (37,306) + `parcel_map.html`; parcel_url N/A (no stable deep link) |
| Artifact scan (Gate 5) | Condo $0-land artifact fixed by imputation (sensitivity printed). Built categories show plausible building shares (0.57–0.89). Parking (median building share 0.02) and vacant sit high under A for a real reason (land-only), not a placeholder. The net taxable base reconciles to −0.71% of the roll |
| OO SFH median change | A −2.8% · B +82.6% (of which about +70 pts is declassification) |
| Vacant land median change | A +64.2% · B +80.7% |

## Limitations

- **Condo land is imputed.** The assessor carries $0 land on every condo unit. Land comes from
  neighborhood median land shares of comparable non-condo parcels. The condo results are
  therefore indicative, and the ±25% sensitivity is printed in Gate 5.
- **312 roll parcels have no CAMA polygon** (about 1.0% of billed tax) and are not modeled.
- **TSA and 8%-law parcels are held at current tax**, since their bills are negotiated or
  statutory.
- **Surface parking comes from OpenStreetMap coverage**, because the assessor has no parking use
  code. That undercounts lots OSM hasn't mapped.
- Legality: RIGL 44-5-11.18 caps are framed on single class rates. The component-wise
  cross-check above is information only.